This notebook focuses on early-stage cleaning, merging, duplicate handling, monthly usage construction, and building-type consolidation. Final validation checks, feature selection, and modeling-specific preprocessing will be handled in later notebooks.

# 01 Data Cleaning

This notebook sets up the initial cleaned datasets for the project using the workbook `annual-energy-consumption-data-2024_cleaning.xlsx`.

For now, the notebook does four things:
1. Loads the Excel workbook with pandas.
2. Creates a `properties` dataframe from the `Properties` sheet using only the required columns.
3. Creates separate gas and electric meter-entry dataframes using only the required columns.
4. Merges the property metadata into the gas and electric dataframes using `Property Name` and `Portfolio Manager ID`.

## Import libraries and define the file path

This cell imports pandas, sets the workbook path, and confirms the available sheet names before we start selecting columns.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

file_path = Path("../data/raw/annual-energy-consumption-data-2024_cleaning.xlsx")

excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

['Properties', 'Meter Entries - Gas', 'Meter Entries - Electric']

## Create the `properties` dataframe

From the first worksheet, `Properties`, we keep only these columns:
- `A`: `Property Name`
- `B`: `Portfolio Manager ID`
- `H`: `Property Type - Self-Selected`
- `I`: `Gross Floor Area`

These are the property-level fields needed later for analysis and modeling.

In [2]:
properties = pd.read_excel(
    file_path,
    sheet_name="Properties",
    usecols="A,B,H,I",
)

properties.head()

,Property Name,Portfolio Manager ID,Property Type - Self-Selected,Gross Floor Area
0,F.J. Horgan Water Treatment Plant,35000838,Drinking Water Treatment & Distribution,325447
1,Island Water Treatment Plant,35000839,Drinking Water Treatment & Distribution,64196
2,W.H. Johnston Pumping Station,35000836,Drinking Water Treatment & Distribution,1744
3,West Toronto Pumping Station,35000837,Drinking Water Treatment & Distribution,7739
4,St. Albans Pumping Station,35000834,Drinking Water Treatment & Distribution,3240


## Create the gas meter dataframe

From the second worksheet, `Meter Entries - Gas`, we keep the requested columns:
- `A`: `Property Name`
- `B`: `Portfolio Manager ID`
- `E`: `Meter Type`
- `G`: `Start Date`
- `H`: `End Date`
- `I`: `Usage/Quantity`
- `J`: `Usage Units`
- `K`: `Cost ($)`

This dataframe is named `gas_entries`.

In [3]:
gas_entries = pd.read_excel(
    file_path,
    sheet_name="Meter Entries - Gas",
    usecols="A,B,E,G,H,I,J,K",
)

gas_entries.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($)
0,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-01-01,2024-02-01,23839.31,cm (cubic meters),10317.96
1,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-02-01,2024-03-01,18735.80,cm (cubic meters),8181.12
2,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-03-01,2024-04-01,14082.81,cm (cubic meters),7422.93
3,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-04-01,2024-05-01,11537.50,cm (cubic meters),5674.99
4,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-05-01,2024-06-01,7784.48,cm (cubic meters),3675.73


## Create the electric meter dataframe

From the third worksheet, `Meter Entries - Electeric`, we keep the requested columns:
- `A`: `Property Name`
- `B`: `Portfolio Manager ID`
- `E`: `Meter Type`
- `G`: `Start Date`
- `H`: `End Date`
- `I`: `Usage/Quantity`
- `J`: `Usage Units`
- `K`: `Cost ($)`

This dataframe is named `electric_entries`.

In [4]:
electric_entries = pd.read_excel(
    file_path,
    sheet_name="Meter Entries - Electric",
    usecols="A,B,E,G,H,I,J,K",
)

electric_entries.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($)
0,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-01-01,2024-02-01,3294191.84,kWh (thousand Watt-hours),335409.93
1,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-02-01,2024-03-01,3188060.05,kWh (thousand Watt-hours),300680.85
2,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-03-01,2024-04-01,2971166.22,kWh (thousand Watt-hours),273185.01
3,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-04-01,2024-05-01,2861591.94,kWh (thousand Watt-hours),255243.37
4,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-05-01,2024-06-01,3142472.63,kWh (thousand Watt-hours),285188.75


## Merge property information into the gas and electric dataframes

Now we attach the property-level information to both meter-entry datasets.

The merge uses both:
- `Property Name`
- `Portfolio Manager ID`

Using both columns makes the join explicit and keeps the property information aligned with the matching meter records.

In [5]:
merge_keys = ["Property Name", "Portfolio Manager ID"]

gas_with_properties = gas_entries.merge(
    properties,
    on=merge_keys,
    how="left",
)

gas_with_properties.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($),Property Type - Self-Selected,Gross Floor Area
0,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-01-01,2024-02-01,23839.31,cm (cubic meters),10317.96,Drinking Water Treatment & Distribution,325447
1,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-02-01,2024-03-01,18735.80,cm (cubic meters),8181.12,Drinking Water Treatment & Distribution,325447
2,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-03-01,2024-04-01,14082.81,cm (cubic meters),7422.93,Drinking Water Treatment & Distribution,325447
3,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-04-01,2024-05-01,11537.50,cm (cubic meters),5674.99,Drinking Water Treatment & Distribution,325447
4,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-05-01,2024-06-01,7784.48,cm (cubic meters),3675.73,Drinking Water Treatment & Distribution,325447


In [6]:
electric_with_properties = electric_entries.merge(
    properties,
    on=merge_keys,
    how="left",
)

electric_with_properties.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($),Property Type - Self-Selected,Gross Floor Area
0,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-01-01,2024-02-01,3294191.84,kWh (thousand Watt-hours),335409.93,Drinking Water Treatment & Distribution,325447
1,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-02-01,2024-03-01,3188060.05,kWh (thousand Watt-hours),300680.85,Drinking Water Treatment & Distribution,325447
2,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-03-01,2024-04-01,2971166.22,kWh (thousand Watt-hours),273185.01,Drinking Water Treatment & Distribution,325447
3,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-04-01,2024-05-01,2861591.94,kWh (thousand Watt-hours),255243.37,Drinking Water Treatment & Distribution,325447
4,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-05-01,2024-06-01,3142472.63,kWh (thousand Watt-hours),285188.75,Drinking Water Treatment & Distribution,325447


## Remove duplicate rows from the merged dataframes

After merging, some rows in gas_with_properties and electric_with_properties are duplicated.

This section removes fully duplicated rows from both merged dataframes and then prints the updated shapes so we can confirm the result.

In [7]:
gas_duplicates_before = gas_with_properties.duplicated().sum()
electric_duplicates_before = electric_with_properties.duplicated().sum()

gas_with_properties = gas_with_properties.drop_duplicates().reset_index(drop=True)
electric_with_properties = electric_with_properties.drop_duplicates().reset_index(drop=True)

print("Duplicate rows removed from gas_with_properties:", gas_duplicates_before)
print("Duplicate rows removed from electric_with_properties:", electric_duplicates_before)

print("\nNew shapes after removing duplicates:")
print("gas_with_properties:", gas_with_properties.shape)
print("electric_with_properties:", electric_with_properties.shape)

Duplicate rows removed from gas_with_properties: 3512
Duplicate rows removed from electric_with_properties: 4578

New shapes after removing duplicates:
gas_with_properties: (8522, 10)
electric_with_properties: (19832, 10)


## Check building type distribution across all prepared dataframes

This cell creates a dictionary of the three current dataframes:

- `properties`
- `gas_with_properties`
- `electric_with_properties`

It then loops through each dataframe and displays a frequency table for `Property Type - Self-Selected`, including missing values (`dropna=False`).

For each dataframe, the output shows:

- the building type category
- the number of rows in that category (`Row Count`)

This is a quick validation step to compare building type coverage before further transformations.

In [8]:
dataframes_with_building_type = {
    'properties': properties,
    'gas_with_properties': gas_with_properties,
    'electric_with_properties': electric_with_properties,
}

for df_name, df in dataframes_with_building_type.items():
    print(f'\n{df_name} building type counts:')
    display(
        df['Property Type - Self-Selected']
        .value_counts(dropna=False)
        .rename_axis('Property Type - Self-Selected')
        .reset_index(name='Row Count')
    )


properties building type counts:


,Property Type - Self-Selected,Row Count
0,Other - Public Services,880
1,Transportation Terminal/Station,169
2,Parking,138
3,Fire Station,92
4,Library,90
5,Community Center and Social Meeting Hall,77
6,Office,53
7,Other - Recreation,51
8,Wastewater Treatment Plant,48
9,Police Station,37



gas_with_properties building type counts:


,Property Type - Self-Selected,Row Count
0,Other - Public Services,3084
1,Fire Station,1080
2,Library,993
3,Community Center and Social Meeting Hall,803
4,Other - Recreation,534
5,Office,516
6,Police Station,379
7,Indoor Arena,318
8,Transportation Terminal/Station,214
9,Other - Entertainment/Public Assembly,205



electric_with_properties building type counts:


,Property Type - Self-Selected,Row Count
0,Other - Public Services,9859
1,Transportation Terminal/Station,1962
2,Parking,1648
3,Fire Station,1092
4,Library,1052
5,Community Center and Social Meeting Hall,924
6,Other - Recreation,600
7,Office,598
8,Wastewater Treatment Plant,564
9,Police Station,421


## Group detailed building types into broader categories

This cell standardizes building type labels across all prepared dataframes.

It does the following:

1. Defines `building_type_mapping`, a dictionary that maps detailed values in  
    `Property Type - Self-Selected` (for example, `Fire Station`, `Library`, `Office`) to broader groups (for example, `Public Safety`, `Education & Social Services`, `Administrative / Office`).

2. Stores the target column name in `type_column` and collects the three dataframes to update in `dataframes_to_update`:
    - `properties`
    - `gas_with_properties`
    - `electric_with_properties`

3. Loops through each dataframe and applies `.replace(building_type_mapping)` to the building type column.

4. Prints a confirmation message after the updates are applied.

This creates consistent building-type categories for downstream analysis.

In [9]:
building_type_mapping = {
    'Fire Station': 'Public Safety',
    'Police Station': 'Public Safety',
    'Indoor Arena': 'Recreation & Community',
    'Community Center and Social Meeting Hall': 'Recreation & Community',
    'Other - Recreation': 'Recreation & Community',
    'Other - Entertainment/Public Assembly': 'Recreation & Community',
    'Office': 'Administrative / Office',
    'Mixed Use Property': 'Administrative / Office',
    'Library': 'Education & Social Services',
    'Pre-school/Daycare': 'Education & Social Services',
    'Residential Care Facility': 'Education & Social Services',
    'Parking': 'Transportation',
    'Transportation Terminal/Station': 'Transportation',
    'Drinking Water Treatment & Distribution': 'Utilities / Infrastructure',
    'Wastewater Treatment Plant': 'Utilities / Infrastructure',
    'Other': 'Other-Public Services',
    'Other - Public Services': 'Other-Public Services',
}

type_column = 'Property Type - Self-Selected'
dataframes_to_update = [properties, gas_with_properties, electric_with_properties]

for df in dataframes_to_update:
    df[type_column] = df[type_column].replace(building_type_mapping)

print('Building types were grouped into the new broader categories for all three dataframes.')

Building types were grouped into the new broader categories for all three dataframes.


## Count rows for each new building type

Now that the detailed property types have been grouped into broader categories, this section counts how many rows belong to each new building type in properties, gas_with_properties, and electric_with_properties.

In [10]:
grouped_type_dataframes = {
    'properties': properties,
    'gas_with_properties': gas_with_properties,
    'electric_with_properties': electric_with_properties,
}

for df_name, df in grouped_type_dataframes.items():
    print(f'\n{df_name} grouped building type counts:')
    display(
        df[type_column]
        .value_counts(dropna=False)
        .rename_axis(type_column)
        .reset_index(name='Row Count')
    )


properties grouped building type counts:


,Property Type - Self-Selected,Row Count
0,Other-Public Services,881
1,Transportation,307
2,Recreation & Community,178
3,Public Safety,129
4,Education & Social Services,111
5,Utilities / Infrastructure,71
6,Administrative / Office,54



gas_with_properties grouped building type counts:


,Property Type - Self-Selected,Row Count
0,Other-Public Services,3084
1,Recreation & Community,1860
2,Public Safety,1459
3,Education & Social Services,1221
4,Administrative / Office,516
5,Transportation,286
6,Utilities / Infrastructure,96



electric_with_properties grouped building type counts:


,Property Type - Self-Selected,Row Count
0,Other-Public Services,9867
1,Transportation,3610
2,Recreation & Community,2100
3,Public Safety,1513
4,Education & Social Services,1304
5,Utilities / Infrastructure,840
6,Administrative / Office,598


## Remove selected portfolio manager IDs from gas data

This cell removes rows from `gas_with_properties` where `Portfolio Manager ID` is `35000492` or `34999500`, then returns the updated dataframe shape.

In [11]:
portfolio_manager_ids_to_remove = [35000492, 34999500]

gas_with_properties = gas_with_properties[
    ~gas_with_properties['Portfolio Manager ID'].isin(portfolio_manager_ids_to_remove)
].reset_index(drop=True)

gas_with_properties.shape

(8497, 10)

## Build monthly gas and electricity usage dataframes

This section converts each billing-level dataframe into a monthly dataframe. It parses the billing dates, removes invalid records, computes daily usage using an exclusive end date, splits each billing period across the calendar months it overlaps, sums the prorated usage for each building-month, and only extrapolates to a full-month value when the combined billings do not cover every day in that month.

In [12]:
def split_billing_record_across_months(record, start_col="Start Date", end_col="End Date"):
    month_rows = []
    start_date = record[start_col]
    end_date = record[end_col]
    last_covered_day = end_date - pd.Timedelta(days=1)

    for month_period in pd.period_range(start=start_date.to_period("M"), end=last_covered_day.to_period("M"), freq="M"):
        month_start = month_period.to_timestamp(how="start")
        next_month_start = (month_period + 1).to_timestamp(how="start")
        overlap_start = max(start_date, month_start)
        overlap_end = min(end_date, next_month_start)
        covered_days = (overlap_end - overlap_start).days

        if covered_days <= 0:
            continue

        month_rows.append(
            {
                "Property Name": record["Property Name"],
                "Portfolio Manager ID": record["Portfolio Manager ID"],
                "Property Type - Self-Selected": record.get("Property Type - Self-Selected"),
                "Gross Floor Area": record.get("Gross Floor Area"),
                "property_order": record["property_order"],
                "month": month_start,
                "overlap_start": overlap_start,
                "overlap_end": overlap_end,
                "covered_days_in_month": covered_days,
                "days_in_month": month_period.days_in_month,
                "prorated_usage": record["daily_usage"] * covered_days,
            }
        )

    return month_rows


def count_unique_covered_days(intervals):
    if not intervals:
        return 0

    sorted_intervals = sorted(intervals, key=lambda interval: interval[0])
    total_days = 0
    current_start, current_end = sorted_intervals[0]

    for next_start, next_end in sorted_intervals[1:]:
        if next_start <= current_end:
            current_end = max(current_end, next_end)
        else:
            total_days += (current_end - current_start).days
            current_start, current_end = next_start, next_end

    total_days += (current_end - current_start).days
    return total_days


def build_monthly_usage_dataframe(
    billing_df,
    start_col="Start Date",
    end_col="End Date",
    usage_col="Usage/Quantity",
):
    working_df = billing_df.copy()
    working_df[start_col] = pd.to_datetime(working_df[start_col], errors="coerce")
    working_df[end_col] = pd.to_datetime(working_df[end_col], errors="coerce")
    working_df[usage_col] = pd.to_numeric(working_df[usage_col], errors="coerce")

    # Preserve the first-seen property order from the original billing dataframe.
    property_order_df = (
        billing_df[["Property Name", "Portfolio Manager ID"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    property_order_df["property_order"] = property_order_df.index
    working_df = working_df.merge(
        property_order_df,
        on=["Property Name", "Portfolio Manager ID"],
        how="left",
    )

    # End dates are treated as exclusive so the end day can belong to the next billing period.
    working_df["billing_days"] = (working_df[end_col] - working_df[start_col]).dt.days

    valid_mask = (
        working_df[start_col].notna()
        & working_df[end_col].notna()
        & (working_df[end_col] > working_df[start_col])
        & working_df[usage_col].notna()
        & (working_df[usage_col] >= 0)
        & (working_df["billing_days"] > 0)
    )

    valid_df = working_df.loc[valid_mask].copy()
    valid_df["daily_usage"] = valid_df[usage_col] / valid_df["billing_days"]

    monthly_rows = []
    for record in valid_df.to_dict(orient="records"):
        monthly_rows.extend(split_billing_record_across_months(record, start_col=start_col, end_col=end_col))

    output_columns = [
        "Property Name",
        "Portfolio Manager ID",
        "Property Type - Self-Selected",
        "Gross Floor Area",
        "month",
        "normalized_monthly_usage",
    ]

    if not monthly_rows:
        return pd.DataFrame(columns=output_columns)

    monthly_detail_df = pd.DataFrame(monthly_rows)
    group_columns = ["Property Name", "Portfolio Manager ID", "month"]

    monthly_usage_df = (
        monthly_detail_df.groupby(group_columns, dropna=False, sort=False)
        .agg(
            {
                "Property Type - Self-Selected": "first",
                "Gross Floor Area": "first",
                "property_order": "first",
                "days_in_month": "first",
                "prorated_usage": "sum",
            }
        )
        .reset_index()
    )

    covered_days_by_month = (
        monthly_detail_df.groupby(group_columns, dropna=False, sort=False)
        .apply(
            lambda group: count_unique_covered_days(
                list(zip(group["overlap_start"], group["overlap_end"]))
            )
        )
        .rename("covered_days_in_month")
        .reset_index()
    )

    monthly_usage_df = monthly_usage_df.merge(covered_days_by_month, on=group_columns, how="left")
    monthly_usage_df["normalized_monthly_usage"] = monthly_usage_df["prorated_usage"]

    partial_coverage_mask = monthly_usage_df["covered_days_in_month"] < monthly_usage_df["days_in_month"]
    monthly_usage_df.loc[partial_coverage_mask, "normalized_monthly_usage"] = (
        monthly_usage_df.loc[partial_coverage_mask, "prorated_usage"]
        * monthly_usage_df.loc[partial_coverage_mask, "days_in_month"]
        / monthly_usage_df.loc[partial_coverage_mask, "covered_days_in_month"]
    )

    monthly_usage_df = monthly_usage_df.sort_values(
        ["property_order", "month"],
        kind="stable"
    ).reset_index(drop=True)

    monthly_usage_df = monthly_usage_df[output_columns]

    return monthly_usage_df


gas_monthly_usage = build_monthly_usage_dataframe(gas_with_properties)
electric_monthly_usage = build_monthly_usage_dataframe(electric_with_properties)

print("gas_monthly_usage shape:", gas_monthly_usage.shape)
display(gas_monthly_usage.head())

print("electric_monthly_usage shape:", electric_monthly_usage.shape)
display(electric_monthly_usage.head())

C:\Users\Mohsen Pasdar\AppData\Local\Temp\ipykernel_16496\1667136208.py:129: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


gas_monthly_usage shape: (8436, 6)


C:\Users\Mohsen Pasdar\AppData\Local\Temp\ipykernel_16496\1667136208.py:129: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,Property Name,Portfolio Manager ID,Property Type - Self-Selected,Gross Floor Area,month,normalized_monthly_usage
0,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-01-01,23839.31
1,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-02-01,18735.80
2,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-03-01,14082.81
3,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-04-01,11537.50
4,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-05-01,7784.48


electric_monthly_usage shape: (19710, 6)


,Property Name,Portfolio Manager ID,Property Type - Self-Selected,Gross Floor Area,month,normalized_monthly_usage
0,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-01-01,3294191.84
1,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-02-01,3188060.05
2,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-03-01,2971166.22
3,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-04-01,2861591.94
4,F.J. Horgan Water Treatment Plant,35000838,Utilities / Infrastructure,325447,2024-05-01,3142472.63


## Check for incomplete monthly coverage

This section checks whether each property has one monthly record for every month of the year in the monthly gas and electricity datasets. It counts how many months are available for each property and lists any missing months for properties that do not have all 12 months.

In [13]:
def check_property_month_completeness(monthly_df, df_name):
    month_check_df = monthly_df.copy()
    month_check_df["month"] = pd.to_datetime(month_check_df["month"], errors="coerce")
    month_check_df = month_check_df.dropna(subset=["Portfolio Manager ID", "month"])

    property_order = (
        month_check_df[["Property Name", "Portfolio Manager ID"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    property_order["property_order"] = property_order.index

    month_counts = (
        month_check_df.groupby(["Property Name", "Portfolio Manager ID"], dropna=False, sort=False)["month"]
        .nunique()
        .reset_index(name="months_available")
    )

    incomplete_properties = month_counts[month_counts["months_available"] < 12].copy()

    if incomplete_properties.empty:
        print(f"{df_name}: all properties have 12 months of data.")
        return incomplete_properties

    missing_months = (
        month_check_df.groupby(["Property Name", "Portfolio Manager ID"], dropna=False, sort=False)["month"]
        .apply(
            lambda months: [
                pd.Timestamp(2000, month_number, 1).strftime("%B")
                for month_number in range(1, 13)
                if month_number not in set(months.dt.month)
            ]
        )
        .reset_index(name="missing_months")
    )

    incomplete_properties = incomplete_properties.merge(
        missing_months,
        on=["Property Name", "Portfolio Manager ID"],
        how="left",
    )
    incomplete_properties = incomplete_properties.merge(
        property_order,
        on=["Property Name", "Portfolio Manager ID"],
        how="left",
    )
    incomplete_properties = incomplete_properties.sort_values(
        ["property_order", "Portfolio Manager ID"],
        kind="stable",
    ).reset_index(drop=True)

    print(f"{df_name}: {incomplete_properties.shape[0]} properties do not have 12 months of data.")
    display(incomplete_properties[["Property Name", "Portfolio Manager ID", "months_available", "missing_months"]])

    return incomplete_properties[["Property Name", "Portfolio Manager ID", "months_available", "missing_months"]]


gas_incomplete_months = check_property_month_completeness(gas_monthly_usage, "gas_monthly_usage")
electric_incomplete_months = check_property_month_completeness(electric_monthly_usage, "electric_monthly_usage")

gas_monthly_usage: 3 properties do not have 12 months of data.


,Property Name,Portfolio Manager ID,months_available,missing_months
0,222 Spadina Ave,35000433,10,"[November, December]"
1,Merton Yard & Sprint Office,35000517,6,"[June, July, August, October, November, December]"
2,Centennial Library,35000307,8,"[September, October, November, December]"


electric_monthly_usage: 10 properties do not have 12 months of data.


,Property Name,Portfolio Manager ID,months_available,missing_months
0,222 Spadina Ave,35000433,9,"[October, November, December]"
1,558 Wilson Ave,35000570,8,"[September, October, November, December]"
2,554 Queen St W Pole 186,35000638,5,"[June, July, August, September, October, Novem..."
3,128 York St Pole 50,35000621,5,"[June, July, August, September, October, Novem..."
4,Islington Subway Stn,35000700,6,"[July, August, September, October, November, D..."
5,Humber Loop,35000696,2,"[March, April, May, June, July, August, Septem..."
6,Old Mill Sewage Pumping Station,35000806,1,"[January, February, March, April, May, June, J..."
7,Clarence Park Encampment Trailer,35490439,8,"[January, February, March, April]"
8,Carpark 89,35000252,2,"[March, April, May, June, July, August, Septem..."
9,Centennial Library,35000307,8,"[September, October, November, December]"


## Remove properties with incomplete monthly coverage

The properties identified as having incomplete monthly records are removed from the monthly gas and electricity dataframes so the remaining datasets contain only properties with full-year monthly coverage.

In [14]:
incomplete_gas_property_ids = [35000433, 35000517, 35000307]
incomplete_electric_property_ids = [35000433, 35000570, 35000638, 35000621, 35000700, 35000696, 35000806, 35490439, 35000252, 35000307]

gas_monthly_usage = gas_monthly_usage[
    ~gas_monthly_usage["Portfolio Manager ID"].isin(incomplete_gas_property_ids)
].reset_index(drop=True)

electric_monthly_usage = electric_monthly_usage[
    ~electric_monthly_usage["Portfolio Manager ID"].isin(incomplete_electric_property_ids)
].reset_index(drop=True)

print("Updated gas_monthly_usage shape:", gas_monthly_usage.shape)
print("Updated electric_monthly_usage shape:", electric_monthly_usage.shape)

Updated gas_monthly_usage shape: (8412, 6)
Updated electric_monthly_usage shape: (19656, 6)


## Export cleaned dataframes to CSV

This section saves the cleaned monthly gas and electricity dataframes, along with the cleaned `properties` dataframe, as CSV files in the `../data/clean/` folder so they can be used in later analysis steps.

In [15]:
export_path = Path("../data/clean")
export_path.mkdir(parents=True, exist_ok=True)

gas_monthly_usage.to_csv(export_path / "gas_monthly_usage.csv", index=False)
electric_monthly_usage.to_csv(export_path / "electric_monthly_usage.csv", index=False)
properties.to_csv(export_path / "properties.csv", index=False)

print("Exported files:")
print(export_path / "gas_monthly_usage.csv")
print(export_path / "electric_monthly_usage.csv")
print(export_path / "properties.csv")

Exported files:
..\data\clean\gas_monthly_usage.csv
..\data\clean\electric_monthly_usage.csv
..\data\clean\properties.csv
